In [1]:
import s3fs
import scipy
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import matplotlib.pyplot as plt
from ibicus.debias import QuantileMapping

from srm import generate_example_data

In [2]:
algorithm = "BCSD_parametric"
mapping_type = "parametric"
data_version = "v0.2"
dir_out = "s3://carbonplan-srm/output/" + data_version + "/"

In [ ]:
def make_figure(dict_all, varname, ilat = -33.9221,ilon = 18.4231, unitconv = 1):
    def plot_pdf(source="era5", label="ERA5", color='k',linewidth=2,alpha=1, linestyle='-', latname="latitude"):
        if latname=="latitude":
            data_to_plot=dict_all[source].sel(latitude=ilat, longitude=ilon, method="nearest")*unitconv
        elif latname=="lat":
            data_to_plot=dict_all[source].sel(lat=ilat, lon=ilon, method="nearest")*unitconv
        minval=np.nanmin(data_to_plot)
        maxval=np.nanmax(data_to_plot)
        sns.kdeplot(data_to_plot, clip=(minval, maxval), 
                    color=color,linewidth=linewidth,alpha=alpha, linestyle=linestyle, label=label)
    
    plt.figure()
    plot_pdf(source="era5", label="ERA5", color='k',linewidth=6,alpha=0.3)

    plot_pdf(source="model_hist", label="Model historical", color='k', linestyle='--', latname="lat")
    plot_pdf(source="g61pt5k", label="G6-1.5K", color='blue', linestyle='--', latname="lat")
    plot_pdf(source="ssp245", label="SSP2-4.5", color='orange', linestyle='--', latname="lat")

    plot_pdf(source="model_hist_debiased_downscaled", label="Model historical (downscaled)", color='k', linestyle='-')
    plot_pdf(source="g61pt5k_debiased_downscaled", label="G6-1.5K (downscaled)", color='blue', linestyle='-')
    plot_pdf(source="ssp245_debiased_downscaled", label="SSP2-4.5 (downscaled)", color='orange', linestyle='-')

    plt.title(varname)
    if varname=="pr":
        plt.xlim([-1,5])
        plt.axvline(x=0,linestyle='--',color='k')
    plt.legend(fontsize=7)
    plt.savefig(varname+"_"+algorithm+".png")

In [ ]:
for varname in ["tas", "tasmax", "tasmin", "rsds", "pr"]:
    # Load raw data
    dict_all = generate_example_data.get_data(varname=varname)

    # Patch fix: change all data < 0 to 0
    if varname == "pr":
        dict_all["era5_clipped"] = dict_all["era5_clipped"].where(
            dict_all["era5_clipped"] > 0, 0
        )
        dict_all["era5"] = dict_all["era5"].where(dict_all["era5"] > 0, 0)
        dict_all["era5_coarse"] = dict_all["era5_coarse"].where(
            dict_all["era5_coarse"] > 0, 0
        )

    # Construct debiaser
    if varname == "rsds":
        debiaser = QuantileMapping(variable=varname, mapping_type=mapping_type, distribution=scipy.stats.norm)
        
    else:
        debiaser = QuantileMapping.from_variable(
            variable=varname, mapping_type=mapping_type
        )

    # Debias simulations
    dict_all = generate_example_data.debias_simulations(debiaser, dict_all)

    # Downscale simulations
    error_map = generate_example_data.calculate_error_map(
        obs_coarse=dict_all["era5_coarse"], obs_fine=dict_all["era5"], dict_all=dict_all
    )

    for scenario in ["ssp245", "g61pt5k", "model_hist"]:
        dict_all[scenario + "_debiased_downscaled"] = (
            generate_example_data.downscale_from_coarse(
                da=dict_all[scenario + "_debiased"],
                error_map=error_map,
                fine_grid=dict_all["era5"],
            )
        )

    # Save downscaled and debiased output
    for scenario in ["ssp245", "g61pt5k", "model_hist"]:
        ds = dict_all[scenario + "_debiased_downscaled"].to_dataset(name=varname)
        ds["ensemble_member"] = ds["ensemble_member"].astype("object")
        fname = varname + "_" + scenario + "_" + algorithm + ".nc"

        # Write to local file
        ds.to_netcdf(fname)

        # Upload to S3
        fs = s3fs.S3FileSystem(anon=False)
        fs.put(fname, dir_out + fname)

        print(fname + " successfully written to S3!")

    if varname=='pr':
        make_figure(dict_all=dict_all, varname=varname, unitconv=86400)
    else:
        make_figure(dict_all=dict_all, varname=varname)

100%|██████████| 620/620 [00:00<00:00, 1641.80it/s]


tas_ssp245_BCSD_parametric.nc successfully written to S3!
tas_g61pt5k_BCSD_parametric.nc successfully written to S3!
tas_model_hist_BCSD_parametric.nc successfully written to S3!


/opt/coiled/env/lib/python3.13/site-packages/ibicus/debias/_quantile_mapping.py:218: UserWarning: The default settings for variable tasmax in debiaser QuantileMapping are currently still experimental and may not have been evaluated in the peer-reviewed literature. Please review the results with care!
  return super()._from_variable(
100%|██████████| 620/620 [00:00<00:00, 1613.91it/s]


tasmax_ssp245_BCSD_parametric.nc successfully written to S3!
tasmax_g61pt5k_BCSD_parametric.nc successfully written to S3!
tasmax_model_hist_BCSD_parametric.nc successfully written to S3!


/opt/coiled/env/lib/python3.13/site-packages/ibicus/debias/_quantile_mapping.py:218: UserWarning: The default settings for variable tasmin in debiaser QuantileMapping are currently still experimental and may not have been evaluated in the peer-reviewed literature. Please review the results with care!
  return super()._from_variable(
100%|██████████| 620/620 [00:00<00:00, 1648.67it/s]


tasmin_ssp245_BCSD_parametric.nc successfully written to S3!
tasmin_g61pt5k_BCSD_parametric.nc successfully written to S3!
tasmin_model_hist_BCSD_parametric.nc successfully written to S3!


100%|██████████| 620/620 [00:00<00:00, 1740.95it/s]
